# Qwen2.5-7B QLoRA — Colab
GPU seç → hücreleri sırayla çalıştır. Veriyi 2. hücrede seç; eğitim sonunda zip indirilir.

In [ ]:
%pip install -q -U "transformers>=4.51.0" "datasets>=3.3.0" "trl>=0.23.0" "peft>=0.14.0" "accelerate>=1.2.0" "bitsandbytes>=0.45.0"
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
from google.colab import files
import json

DATA_PATH = Path("/content/modelai.jsonl")
if not DATA_PATH.exists():
    DATA_PATH = Path("/content/" + next(iter(files.upload())))  # pencereden modelai.jsonl'yi seç

lines = [l for l in DATA_PATH.read_text(encoding="utf-8").splitlines() if l.strip()]
records = [{"prompt": [{"role": "user", "content": r["user"]}],
            "completion": [{"role": "assistant", "content": r["assistant"]}]}
           for r in map(json.loads, lines)]
print(len(records), "kayıt")

In [ ]:
from datasets import Dataset
train_ds = Dataset.from_list(records).train_test_split(test_size=0.02, seed=42)["train"]
print(len(train_ds), "eğitim kaydı")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
import torch

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                           bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=quant,
                                             torch_dtype=torch.float16, device_map="auto")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

for _, p in model.named_parameters():
    if p.dtype == torch.bfloat16:
        p.data = p.data.to(torch.float16)

args = SFTConfig(output_dir="/content/out", num_train_epochs=2,
    per_device_train_batch_size=1, gradient_accumulation_steps=16, learning_rate=2e-4,
    lr_scheduler_type="cosine", max_grad_norm=0.3,
    optim="paged_adamw_8bit", fp16=False, bf16=False, gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, max_length=1024,
    completion_only_loss=True, save_strategy="epoch", save_total_limit=1,
    logging_steps=20, report_to="none", seed=42, remove_unused_columns=False)

trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds,
                     processing_class=tokenizer,
                     peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                                            bias="none", task_type="CAUSAL_LM",
                                            target_modules="all-linear"))
trainer.train()
trainer.save_model("/content/out")
tokenizer.save_pretrained("/content/out")

In [ ]:
import shutil
from google.colab import files
files.download(shutil.make_archive("/content/qwen25_7b_qlora", "zip", "/content/out"))